In [1]:
import librosa
import numpy as np
import soundfile as sf

import torch
import torchaudio
from transformers import AutoModelForAudioClassification, AutoProcessor
from speechbrain.pretrained import EncoderClassifier

/home/alae/anaconda3/envs/test/lib/python3.9/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# DETECT IF VOICE IS AI OR HUMAN WITHOUT ML/DL

In [54]:
def detect_ai_voice(audio_path):
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    avg_pitch = float(np.mean(pitch_values)) if len(pitch_values) > 0 else 0.0
    spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    energy = librosa.feature.rms(y=y)[0]
    silent_frames = np.sum(energy < 0.01)
    gap_ratio = float(silent_frames / len(energy))
    if avg_pitch < 750 or gap_ratio < 0.5:
        classification = "AI_GENERATED"
        explanation = "Low pitch variation or very few silence gaps detected."
    else:
        classification = "HUMAN"
        explanation = "Natural pitch variation and silence gaps detected."
    return {
        "classification": classification,
        "avg_pitch": round(avg_pitch, 2),
        "spectral_centroid": round(spectral_centroid, 2),
        "gap_ratio": round(gap_ratio, 3),
        "explanation": explanation}

In [55]:
path = "/home/alae/Desktop/journal_rev/rev_ravdess/test/ravdess/Actor_01/03-01-01-01-01-01-01.wav"
result = detect_ai_voice(path)
result

{'classification': 'HUMAN',
 'avg_pitch': 1778.66,
 'spectral_centroid': 2589.8,
 'gap_ratio': 0.971,
 'explanation': 'Natural pitch variation and silence gaps detected.'}

In [56]:
path = "/home/alae/Desktop/env/notbooks/speeches/ElevenLabs_2.mp3"
result = detect_ai_voice(path)
result

{'classification': 'AI_GENERATED',
 'avg_pitch': 1707.98,
 'spectral_centroid': 2358.23,
 'gap_ratio': 0.2,
 'explanation': 'Low pitch variation or very few silence gaps detected.'}

# 🎧 SIMPLE Text → Speech (WAV / MP3)

In [58]:
from gtts import gTTS

text = "This is an artificial intelligence generated voice for testing."
tts = gTTS(text=text, lang="en")
tts.save("ai_voice.mp3")
print("MP3 generated")

MP3 generated


In [65]:
wav, sr = librosa.load("ai_voice.mp3")
sr

22050

_____

In [66]:
from pydub import AudioSegment

wav, sr = librosa.load("ai_voice.mp3")
audio = AudioSegment.from_mp3("ai_voice.mp3")
audio = audio.set_frame_rate(sr).set_channels(1)
audio.export("ai_voice.wav", format="wav")
print("WAV generated")

WAV generated


In [67]:
import os
print("MP3 size:", os.path.getsize("ai_voice.mp3"), "bytes")
print("WAV size:", os.path.getsize("ai_voice.wav"), "bytes")

MP3 size: 38016 bytes
WAV size: 209606 bytes


___

In [68]:
import librosa
import numpy as np

y_mp3, sr = librosa.load("ai_voice.mp3", sr=16000)
y_wav, _  = librosa.load("ai_voice.wav", sr=16000)

f0_mp3 = librosa.yin(y_mp3, fmin=50, fmax=300)
f0_wav = librosa.yin(y_wav, fmin=50, fmax=300)

print("Pitch variance MP3:", np.nanvar(f0_mp3))
print("Pitch variance WAV:", np.nanvar(f0_wav))

Pitch variance MP3: 4614.441915941926
Pitch variance WAV: 4754.658635586024


In [69]:
path = "ai_voice.mp3"
result = detect_ai_voice(path)
result

{'classification': 'AI_GENERATED',
 'avg_pitch': 1706.74,
 'spectral_centroid': 2198.49,
 'gap_ratio': 0.154,
 'explanation': 'Low pitch variation or very few silence gaps detected.'}

In [70]:
path = "ai_voice.wav"
result = detect_ai_voice(path)
result

{'classification': 'AI_GENERATED',
 'avg_pitch': 1695.71,
 'spectral_centroid': 2117.34,
 'gap_ratio': 0.154,
 'explanation': 'Low pitch variation or very few silence gaps detected.'}

___

In [77]:
text = "Bonjour, ceci est une voix générée par intelligence artificielle."
tts = gTTS(text=text, lang="fr")
tts.save("ai_voice_fr.mp3")

In [79]:
path = "ai_voice_fr.mp3"
result = detect_ai_voice(path)
result

{'classification': 'AI_GENERATED',
 'avg_pitch': 1433.99,
 'spectral_centroid': 1964.78,
 'gap_ratio': 0.166,
 'explanation': 'Low pitch variation or very few silence gaps detected.'}

____

In [7]:
from gtts import gTTS

text = "This is an artificial intelligence generated voice for testing."
tts = gTTS(text=text, lang="en", slow=True, tld="co.uk")
tts.save("test_true_uk.mp3")
print("MP3 generated")

MP3 generated


In [8]:
tts = gTTS(text=text, lang="en", slow=False, tld="co.uk")
tts.save("test_false_uk.mp3")
print("MP3 generated")

MP3 generated


In [9]:
tts = gTTS(text=text, lang="en", slow=False, tld="com")
tts.save("test_false_com.mp3")
print("MP3 generated")

MP3 generated


# 🎧 Simple Speech → Text (WAV / MP3)

In [73]:
import whisper
# Load model base
model = whisper.load_model("base")

In [81]:
path_fr = "ai_voice_fr.mp3"
result = model.transcribe(path_fr)
print(result["text"])

/home/alae/anaconda3/envs/test/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Bonjour, ceci est une voix générée par intelligence artificielle.


### detect language

In [85]:
import whisper
import torch

model = whisper.load_model("base")
audio = whisper.load_audio(path_fr)
audio = whisper.pad_or_trim(audio)

mel = whisper.log_mel_spectrogram(audio).to(model.device)
_, probs = model.detect_language(mel)

print(max(probs, key=probs.get))

fr


# LIVE transcription with VOICE ACTIVITY DETECTION (VAD)

In [3]:
import sounddevice as sd
import numpy as np
import whisper
import time
import threading
from IPython.display import clear_output

## stop if 5 seconds of silence

In [4]:
RATE = 44100
CHANNELS = 1
BLOCK_DURATION = 0.5       # seconds per chunk
SILENCE_THRESHOLD = 0.01
SILENCE_LIMIT = 5          # seconds of silence to stop
model = whisper.load_model("base")

In [5]:
stop_flag = {"stop": False}
def wait_for_enter():
    input("Press ENTER to stop recording manually...\n")
    stop_flag["stop"] = True

threading.Thread(target=wait_for_enter, daemon=True).start()
recorded_audio = []
last_speech_time = time.time()

def rms(y):
    return np.sqrt(np.mean(y**2))

print("Start speaking... (press ENTER to stop)")
with sd.InputStream(samplerate=RATE, channels=CHANNELS, dtype='float32',
                    blocksize=int(RATE*BLOCK_DURATION)) as stream:
    while True:
        if stop_flag["stop"]:
            print("\nManual stop triggered!")
            break
        # Read audio chunk
        data, _ = stream.read(int(RATE*BLOCK_DURATION))
        data = data.flatten()
        recorded_audio.append(data)
        # Simple RMS-based VAD
        if rms(data) > SILENCE_THRESHOLD:
            last_speech_time = time.time()
        # Stop if silence
        if time.time() - last_speech_time > SILENCE_LIMIT:
            print("\nNo voice for 5 seconds, stopping...")
            break
        # Live transcription
        audio_np = np.concatenate(recorded_audio)
        try:
            result = model.transcribe(audio_np, fp16=False)
            print("Live Transcript:", result["text"])
        except Exception as e:
            print("Transcription error:", e)

Start speaking... (press ENTER to stop)
Live Transcript: 


Press ENTER to stop recording manually...
 


Live Transcript: 

Manual stop triggered!
